In [1]:
import pandas as pd
from transformers import T5ForConditionalGeneration, T5Tokenizer, Trainer, TrainingArguments

In [2]:
# Load Dataset

train_data = pd.read_csv("/content/data/samsum-train.csv")
validation_data = pd.read_csv("/content/data/samsum-validation.csv")

print(train_data.head())
print(f"Train Shape: {train_data.shape}")
print(f"Validation Shape: {validation_data.shape}")

         id                                           dialogue  \
0  13818513  Amanda: I baked  cookies. Do you want some?\r\...   
1  13728867  Olivia: Who are you voting for in this electio...   
2  13681000  Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...   
3  13730747  Edward: Rachel, I think I'm in ove with Bella....   
4  13728094  Sam: hey  overheard rick say something\r\nSam:...   

                                             summary  
0  Amanda baked cookies and will bring Jerry some...  
1  Olivia and Olivier are voting for liberals in ...  
2  Kim may try the pomodoro technique recommended...  
3  Edward thinks he is in love with Bella. Rachel...  
4  Sam is confused, because he overheard Rick com...  
Train Shape: (14732, 3)
Validation Shape: (818, 3)


In [14]:
# Select random samples for Train and Validation dataset

train_data = train_data.sample(n = 5000, random_state = 42).reset_index(drop = True)
validation_data = validation_data.sample(n = 500, random_state = 42).reset_index(drop = True)

print(f"Train Shape: {train_data.shape}")
print(f"Validation Shape: {validation_data.shape}")

Train Shape: (5000,)
Validation Shape: (500,)


In [4]:
# Clean Text

import re

def clean_text(text):
  text = re.sub(r"\r\n", " ", text)
  text = re.sub(r"\s+", " ", text)
  text = re.sub(r"<.*?>", " ", text)
  text = text.strip().lower()

  return text

train_data["dialogue"] = train_data["dialogue"].apply(clean_text)
train_data["summary"] = train_data["summary"].apply(clean_text)

validation_data["dialogue"] = validation_data["dialogue"].apply(clean_text)
validation_data["summary"] = validation_data["summary"].apply(clean_text)

In [5]:
# Load Tokenzer

tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [6]:
# Apply Tokenization

def preprocess_function(examples):
  inputs = tokenizer(examples["dialogue"], max_length = 512, padding = "max_length", truncation = True)
  targets = tokenizer(examples["summary"], max_length = 175, padding = "max_length", truncation = True)

  inputs["labels"] = targets["input_ids"]

  return inputs

train_data = train_data.apply(preprocess_function, axis = 1)
validation_data = validation_data.apply(preprocess_function, axis = 1)

In [12]:
# Set Model

model = T5ForConditionalGeneration.from_pretrained("t5-small")

training_args = TrainingArguments(
  num_train_epochs = 6,
  per_device_train_batch_size = 8,
  per_device_eval_batch_size = 8,
  warmup_steps = 500,
  weight_decay = 0.01,
  save_steps = 500,
  eval_steps = 50,
  do_eval = True
)

trainer = Trainer(
  model = model,
  args = training_args,
  train_dataset = train_data,
  eval_dataset = validation_data
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [15]:
# Train Model

trainer.train()

Step,Training Loss
500,3.612468
1000,0.344039
1500,0.327500
2000,0.313683
2500,0.310141
3000,0.302298


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss
500,3.612468
1000,0.344039
1500,0.327500
2000,0.313683
2500,0.310141
3000,0.302298
3500,0.304057


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.7550229370117187, metrics={'train_runtime': 1652.4981, 'train_samples_per_second': 18.154, 'train_steps_per_second': 2.269, 'total_flos': 4060254044160000.0, 'train_loss': 0.7550229370117187, 'epoch': 6.0})

In [16]:
# Save Models

model.save_pretrained("./saved_files/summarization_model")
tokenizer.save_pretrained("./saved_files/tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_files/tokenizer/tokenizer_config.json',
 './saved_files/tokenizer/tokenizer.json')

In [32]:
# Install ONNX dependencies

!pip uninstall -y optimum optimum-onnx diffusers
!pip install -q --upgrade "huggingface_hub>=0.34.0" "optimum[onnxruntime]" accelerate sentencepiece


Found existing installation: optimum 2.1.0
Uninstalling optimum-2.1.0:
  Successfully uninstalled optimum-2.1.0
Found existing installation: optimum-onnx 0.1.0
Uninstalling optimum-onnx-0.1.0:
  Successfully uninstalled optimum-onnx-0.1.0
Found existing installation: diffusers 0.42.1
Uninstalling diffusers-0.42.1:
  Successfully uninstalled diffusers-0.42.1


In [33]:
# Export Model to ONNX

!optimum-cli export onnx \
    --model ./saved_files/summarization_model \
    --task text2text-generation-with-past \
    ./saved_files/onnx_export


I0000 00:00:1793648310.318449   21498 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793648315.087571   21498 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
`torch_dtype` is deprecated! Use `dtype` instead!
/usr/local/lib/python3.13/dist-packages/optimum/exporters/base.py:151: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self._normalized_config = self.NORMALIZED_CONFIG_CLASS(self._config)
/usr/local/lib/python3.13/dist-packages/transformers/models/t5/modeling_t5.py:1271: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, 

In [34]:
import os

export_dir = "./saved_files/onnx_export"

print(os.listdir(export_dir))

['decoder_model.onnx', 'decoder_model_merged.onnx', 'encoder_model.onnx', 'generation_config.json', 'config.json', 'decoder_with_past_model.onnx']


In [35]:
# Use Models

model = T5ForConditionalGeneration.from_pretrained("./saved_files/summarization_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_files/tokenizer")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [36]:
# Build Summarization Model

device = model.device

def summarize_dialogue(dialogue):
  dialogue = clean_text(dialogue)
  # tokenizer return: input_ids, attention_mask
  # tokenizer.encode return: input_ids
  input_ids = tokenizer(dialogue, return_tensors = "pt", max_length = 512, truncation = True)

  inputs = {key: value.to(device) for key, value in input_ids.items()}

  outputs = model.generate(
    input_ids["input_ids"],
    max_length = 175,
    num_beams = 4,
    early_stopping = True,
  )

  summary = tokenizer.decode(outputs[0], skip_special_tokens = True)
  return summary

In [37]:
# Test Sample

origianl_dialogue = """
Character A: Are you reading a manual for the microwave? You've owned it for three years.

Character B: I just discovered it has a mute button. For three years, I've been running across the kitchen at 1:00 AM like a bomb technician to stop the beep at one second.

Character A: You mean you didn't know holding down the zero key turns off the sound?

Character B: You knew this whole time and watched me sprint past you every night?

Character A: Honestly? It was the highlight of my evening.
"""

summary = summarize_dialogue(origianl_dialogue)
print(f"Summary: {summary}")

Summary: character a has owned a microwave manual for three years. he's been running across the kitchen at 1:00 am like a bomb technician to stop the beep at one second.


In [38]:
# Archive Model

import shutil

shutil.make_archive("summarization_model", "zip", "./saved_files")

shutil.make_archive(
    "onnx_summarization_model",
    "zip",
    "./saved_files/onnx_export"
)

from google.colab import files

files.download("summarization_model.zip")
files.download("onnx_summarization_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>